# Red neuronal multicapa (MLP) con PyTorch

**Materiales desarrollados por Matías Barreto, 2025**

**Tecnicatura en Ciencia de Datos - IFTS**

**Asignatura:** Procesamiento de Lenguaje Natural

---

## Objetivo

Dar el salto desde una sola neurona artificial hacia una red multicapa que pueda aprender relaciones no lineales usando PyTorch.

## Resultados de aprendizaje

Al final de este notebook vas a poder:

1. Explicar qué agrega una capa oculta respecto de un perceptrón simple.
2. Diferenciar datos (`tensor`) y parámetros entrenables en PyTorch.
3. Definir un `nn.Module` con capas visibles y separadas.
4. Seguir el ciclo `forward -> loss -> backward -> step`.
5. Evaluar el modelo con una pequeña prueba externa y no solo sobre entrenamiento.

## Relación con el notebook anterior

En `03` programaste una neurona artificial a mano. Ahora vamos a mantener la lógica general, pero delegaremos en PyTorch el cálculo de gradientes y la actualización de parámetros.

## Introducción

El perceptrón simple nos sirvió para entender cómo aprende una neurona. Su límite principal fue claro: solo puede resolver patrones linealmente separables. Una red multicapa agrega capas ocultas y funciones de activación diferenciables, lo que amplía mucho la capacidad del modelo.


---

## 1. Instalación e Importación de PyTorch

En Google Colab, PyTorch ya viene instalado. Para instalación local, visitar: https://pytorch.org

In [ ]:
# Si necesitás instalar PyTorch (descomentá la línea siguiente)
# !pip install torch

# PyTorch: Framework de deep learning
import torch
# torch.nn: Módulo con capas y modelos de redes neuronales
import torch.nn as nn
# torch.optim: Optimizadores (SGD, Adam, etc.)
import torch.optim as optim

# NumPy: Para operaciones numéricas complementarias
import numpy as np

# Matplotlib: Para visualizaciones
import matplotlib.pyplot as plt

# Fijamos semillas para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Ejecutando en CPU (suficiente para este notebook)")

---

## 2. Dataset: análisis de sentimiento en español rioplatense

Usamos un corpus más grande que en el notebook del perceptrón porque una red con más parámetros necesita más ejemplos para mostrar mejor su comportamiento.

Aun así, seguí pensando este dataset como un corpus didáctico: sirve para ver el mecanismo, no para sacar conclusiones de rendimiento de producción.


In [ ]:
# Corpus ampliado de reseñas en español rioplatense
# Etiqueta: 1 = Positivo, 0 = Negativo
frases = [
    # Positivas
    "La verdad, este lugar está bárbaro. Muy recomendable.",
    "Qué buena onda la atención, volvería sin dudarlo.",
    "Me encantó la comida, aunque la música estaba muy fuerte.",
    "Todo excelente. Atención de diez.",
    "Muy conforme con el resultado final.",
    "Superó mis expectativas, gracias.",
    "El mejor asado que probé en mucho tiempo.",
    "Excelente relación precio-calidad, muy recomendable.",
    "La atención fue impecable, muy atentos.",
    "Me gustó mucho el ambiente tranquilo.",

    # Negativas
    "Una porquería de servicio, nunca más vuelvo.",
    "El envío fue lento y el producto llegó dañado. Qué desastre.",
    "Qué estafa, me arrepiento de haber comprado.",
    "No me gustó para nada la experiencia.",
    "No lo recomiendo, mala calidad.",
    "Malísima atención, el mozo tenía mala onda.",
    "Tardaron dos horas en entregar, llegó todo frío.",
    "Me cobraron de más y encima se hicieron los giles.",
    "La carne estaba pasada, casi no se podía comer.",
    "Pésima experiencia, no vuelvo más."
]

# Etiquetas: 1=positivo, 0=negativo
etiquetas = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # 10 positivas
                      0, 0, 0, 0, 0, 0, 0, 0, 0, 0])  # 10 negativas

print(f"Corpus de entrenamiento: {len(frases)} frases")
print(f"Distribución: {np.bincount(etiquetas)} (negativas, positivas)")
print(f"\nPrimeras 3 frases positivas:")
for i in range(3):
    print(f"  {i+1}. {frases[i]}")
print(f"\nPrimeras 3 frases negativas:")
for i in range(10, 13):
    print(f"  {i-9}. {frases[i]}")

---

## 3. Vocabulario y Vectorización

Construimos un vocabulario extendido con palabras clave del corpus.

In [ ]:
# Vocabulario manual con palabras discriminativas
vocabulario = [
    # Positivas
    "bárbaro", "recomendable", "buena", "onda", "encantó",
    "excelente", "conforme", "superó", "expectativas", "mejor",
    "impecable", "gustó", "tranquilo",

    # Negativas
    "porquería", "nunca", "desastre", "estafa", "arrepiento",
    "no", "mala", "malísima", "pésima", "tardaron",
    "frío", "pasada",

    # Neutras/contextuales
    "lugar", "atención", "comida", "servicio", "experiencia"
]

print(f"Vocabulario: {len(vocabulario)} palabras")
print(vocabulario)

In [ ]:
def vectorizar(frase, vocabulario):
    """
    Convierte una frase en un vector binario según el vocabulario.

    Args:
        frase (str): Texto a vectorizar.
        vocabulario (list): Lista de palabras clave.

    Returns:
        np.array: Vector binario (1 si la palabra aparece, 0 si no).
    """
    tokens = frase.lower().split()

    valores_vector = []
    for palabra in vocabulario:
        if palabra in tokens:
            valores_vector.append(1)
        else:
            valores_vector.append(0)

    vector = np.array(valores_vector, dtype=np.float32)
    return vector


filas_vectorizadas = []
for frase in frases:
    fila_vectorizada = vectorizar(frase, vocabulario)
    filas_vectorizadas.append(fila_vectorizada)

X_np = np.array(filas_vectorizadas, dtype=np.float32)
y_np = etiquetas.astype(np.float32).reshape(-1, 1)

print("Matriz de características:")
print(f"Forma de X: {X_np.shape} (muestras x features)")
print(f"Forma de y: {y_np.shape}")
print("
Primera frase vectorizada:")
print(f"Frase: '{frases[0]}'")
print(f"Vector: {X_np[0]}")

palabras_presentes = []
for indice_palabra in range(len(vocabulario)):
    if X_np[0][indice_palabra] == 1:
        palabras_presentes.append(vocabulario[indice_palabra])

print(f"Palabras detectadas: {palabras_presentes}")


---

## 4. Introducción a Tensores en PyTorch

Los **tensores** son el equivalente de NumPy arrays en PyTorch, pero con capacidades adicionales:
- Pueden ejecutarse en GPU
- Registran operaciones para backpropagation automática (autograd)
- Integración nativa con redes neuronales

In [ ]:
# Convertimos NumPy arrays a tensores de PyTorch
# torch.tensor() crea un nuevo tensor copiando los datos
X = torch.tensor(X_np)
y = torch.tensor(y_np)

print("Información de los tensores:")
print("=" * 70)
print(f"X.shape: {X.shape}")
print(f"X.dtype: {X.dtype}")
print(f"X.device: {X.device}  # cpu o cuda")
print(f"X.requires_grad: {X.requires_grad}  # False porque son datos, no parámetros")
print()
print(f"y.shape: {y.shape}")
print(f"y.dtype: {y.dtype}")

print("\n" + "=" * 70)
print("Operaciones con tensores:")
print("-" * 70)

# Operaciones básicas (similares a NumPy)
print(f"Media de X: {X.mean().item():.4f}")
print(f"Desviación estándar de X: {X.std().item():.4f}")
print(f"Suma de y: {y.sum().item():.0f}")

# .item() extrae el valor de un tensor de un solo elemento

---

## 5. Arquitectura de una Red Multicapa (MLP)

Una red feedforward multicapa consta de:

```
INPUT → [CAPA OCULTA 1] → [CAPA OCULTA 2] → ... → OUTPUT
```

### Arquitectura que vamos a implementar:

```
Input Layer          Hidden Layer        Output Layer
(30 features)        (8 neuronas)        (1 neurona)

x₁ ─┐
x₂ ─┤               h₁ ─┐
x₃ ─┤──────────────>h₂ ─┤
... │    ReLU        h₃ ─┤──────────────> y
x₃₀─┘               h₄ ─┤    Sigmoid
                    ... ─┘
                    h₈
```

### Componentes:

1. **Capa de entrada**: 30 features (tamaño del vocabulario)
2. **Capa oculta**: 8 neuronas con activación ReLU
3. **Capa de salida**: 1 neurona con activación Sigmoid (probabilidad 0-1)

### ¿Por qué ReLU?

**ReLU (Rectified Linear Unit):**
$$\text{ReLU}(x) = \max(0, x)$$

**Ventajas sobre escalón:**
- Diferenciable en casi todos lados (excepto x=0)
- No sufre vanishing gradient
- Computacionalmente eficiente
- Introduce no linealidad necesaria para problemas complejos

### ¿Por qué Sigmoid en la salida?

**Sigmoid:**
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**Ventajas:**
- Salida en rango [0, 1]: interpretable como probabilidad
- Diferenciable en todos lados
- Compatible con Binary Cross Entropy Loss

---

## 6. Definición del modelo con `nn.Module`

En PyTorch, todos los modelos heredan de `nn.Module`. En este notebook vamos a evitar una definición demasiado compacta para que cada capa quede visible por separado.


In [ ]:
class MLP(nn.Module):
    """
    Red neuronal multicapa para clasificación binaria de sentimientos.
    """

    def __init__(self, input_size, hidden_size):
        """
        Inicializa las capas de la red.

        Args:
            input_size (int): Cantidad de features de entrada.
            hidden_size (int): Cantidad de neuronas en la capa oculta.
        """
        super().__init__()

        self.capa_oculta = nn.Linear(input_size, hidden_size)
        self.activacion_oculta = nn.ReLU()
        self.capa_salida = nn.Linear(hidden_size, 1)
        self.activacion_salida = nn.Sigmoid()

    def forward(self, x):
        """
        Define el recorrido de los datos por la red.
        """
        salida_oculta = self.capa_oculta(x)
        salida_oculta = self.activacion_oculta(salida_oculta)
        salida_final = self.capa_salida(salida_oculta)
        salida_final = self.activacion_salida(salida_final)
        return salida_final


input_size = len(vocabulario)
hidden_size = 8
modelo = MLP(input_size, hidden_size)

print("Modelo MLP creado:")
print("=" * 70)
print(modelo)
print("
" + "=" * 70)
print("Parámetros del modelo:")
print("-" * 70)

total_params = 0
for _, parametro in modelo.named_parameters():
    if parametro.requires_grad:
        total_params += parametro.numel()

print(f"Total de parámetros entrenables: {total_params}")

for nombre, parametro in modelo.named_parameters():
    shape_parametro = list(parametro.shape)
    print(f"  {nombre:20s} -> shape: {str(shape_parametro):20s} | {parametro.numel():4d} params")

print("
Nota: cada tensor de parámetros se inicializa automáticamente y se ajusta durante el entrenamiento.")


---

## 7. Función de Pérdida (Loss Function)

Para clasificación binaria, usamos **Binary Cross Entropy (BCE)**:

$$\text{BCE} = -\frac{1}{n}\sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

Donde:
- $y_i$: Etiqueta real (0 o 1)
- $\hat{y}_i$: Probabilidad predicha por el modelo

### Intuición:

- Si $y=1$ y $\hat{y}=0.9$: Pérdida baja (buena predicción)
- Si $y=1$ y $\hat{y}=0.1$: Pérdida alta (mala predicción)
- La función es convexa: garantiza convergencia a un mínimo

### ¿Por qué no MSE (Mean Squared Error)?

MSE funciona para regresión, pero para clasificación:
- BCE tiene mejores propiedades de gradiente
- Penaliza predicciones muy erróneas más fuerte
- Es la elección estándar para problemas binarios

In [ ]:
# Definimos la función de pérdida
# BCELoss: Binary Cross Entropy para clasificación binaria
criterio = nn.BCELoss()

print("Función de pérdida: Binary Cross Entropy (BCE)")
print("=" * 70)
print(criterio)

# Ejemplo de cómo funciona BCE
print("\nEjemplos de pérdida:")
print("-" * 70)

# Casos de ejemplo
casos = [
    (1.0, 0.9, "Buena predicción: y=1, pred=0.9"),
    (1.0, 0.5, "Predicción incierta: y=1, pred=0.5"),
    (1.0, 0.1, "Mala predicción: y=1, pred=0.1"),
    (0.0, 0.1, "Buena predicción: y=0, pred=0.1"),
    (0.0, 0.9, "Mala predicción: y=0, pred=0.9"),
]

for y_true, y_pred, descripcion in casos:
    y_tensor = torch.tensor([[y_true]])
    pred_tensor = torch.tensor([[y_pred]])
    loss = criterio(pred_tensor, y_tensor)
    print(f"{descripcion:40s} → Loss: {loss.item():.4f}")

print("\nObservación: A mayor error, mayor pérdida (penalización).")

---

## 8. Optimizador: Adam

El optimizador actualiza los parámetros del modelo basándose en los gradientes calculados por backpropagation.

### Adam (Adaptive Moment Estimation)

Es el optimizador más usado en deep learning moderno porque:
1. **Adaptativo**: Ajusta learning rate por parámetro
2. **Momento**: Usa promedios móviles de gradientes
3. **Robusto**: Funciona bien con hiperparámetros por defecto
4. **Eficiente**: Converge rápido

### Otros optimizadores comunes:

- **SGD (Stochastic Gradient Descent)**: Básico pero efectivo
- **RMSprop**: Precursor de Adam
- **AdaGrad**: Para sparse data

### Learning Rate

El parámetro `lr` (learning rate) controla el tamaño del paso:
- Típicamente: 0.001 - 0.01 para Adam
- Muy alto: Divergencia, no converge
- Muy bajo: Convergencia lenta

In [ ]:
# Definimos el optimizador Adam
# modelo.parameters() retorna todos los parámetros entrenables (pesos y biases)
# lr: learning rate (tasa de aprendizaje)
optimizador = optim.Adam(modelo.parameters(), lr=0.01)

print("Optimizador configurado:")
print("=" * 70)
print(optimizador)
print("\nParámetros que el optimizador va a actualizar:")
print("-" * 70)
for i, param_group in enumerate(optimizador.param_groups):
    print(f"Grupo {i}: {len(param_group['params'])} tensores")
    print(f"  Learning rate: {param_group['lr']}")

---

## 9. Bucle de entrenamiento

El objetivo de este bloque es observar con claridad el ciclo básico de entrenamiento en PyTorch. Más adelante veremos versiones más compactas, pero por ahora conviene dejar cada paso visible.


In [ ]:
# Hiperparámetros de entrenamiento
epocas = 200  # Número de pasadas completas por el dataset

# Listas para guardar el historial
historial_loss = []

print("Iniciando entrenamiento...")
print("=" * 70)

# Ponemos el modelo en modo entrenamiento
# Esto afecta capas como Dropout y BatchNorm (no las usamos aquí, pero es buena práctica)
modelo.train()

for epoca in range(epocas):
    # Paso 1: Forward pass
    # Pasamos los datos por la red
    salida = modelo(X)

    # Paso 2: Calcular pérdida
    loss = criterio(salida, y)

    # Paso 3: Backward pass
    # Limpiamos gradientes de la iteración anterior
    optimizador.zero_grad()

    # Calculamos gradientes con backpropagation
    # PyTorch calcula automáticamente ∂loss/∂w para todos los parámetros
    loss.backward()

    # Paso 4: Actualizar parámetros
    # El optimizador ajusta los pesos usando los gradientes
    optimizador.step()

    # Guardamos la pérdida para visualización
    historial_loss.append(loss.item())

    # Imprimimos progreso cada 10 épocas
    if (epoca + 1) % 10 == 0:
        print(f"Época {epoca+1:3d}/{epocas}: Loss = {loss.item():.4f}")

print("\n" + "=" * 70)
print("Entrenamiento completado.")
print(f"Pérdida final: {historial_loss[-1]:.4f}")

---

## 10. Visualización del Aprendizaje

Graficamos la curva de aprendizaje (loss vs. épocas) para visualizar la convergencia.

In [ ]:
# Gráfico de la pérdida durante el entrenamiento
plt.figure(figsize=(10, 5))
plt.plot(historial_loss, linewidth=2)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Loss (Binary Cross Entropy)', fontsize=12)
plt.title('Curva de Aprendizaje del MLP', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretación de la curva:")
print("=" * 70)
print("- Curva descendente: El modelo está aprendiendo")
print("- Convergencia: La pérdida se estabiliza en un valor bajo")
print("- Si oscila mucho: Learning rate probablemente muy alto")
print("- Si baja muy lento: Learning rate probablemente muy bajo")
print("\nEsta curva muestra que el modelo convergió correctamente.")

---

## 11. Evaluación: entrenamiento y prueba externa pequeña

Primero vamos a mirar el ajuste sobre las frases del corpus. Después vamos a usar una pequeña prueba externa con frases nuevas y etiquetas definidas a mano.

No es una evaluación estadísticamente robusta, pero sí evita la mala práctica de quedarse solo con el rendimiento sobre entrenamiento.


In [ ]:
modelo.eval()

with torch.no_grad():
    predicciones_entrenamiento = modelo(X)
    clases_entrenamiento = (predicciones_entrenamiento >= 0.5).float()

aciertos_entrenamiento = (clases_entrenamiento == y).sum().item()
total_entrenamiento = len(y)
accuracy_entrenamiento = aciertos_entrenamiento / total_entrenamiento

print("Control sobre entrenamiento:")
print("=" * 70)
for i in range(len(frases)):
    prob = predicciones_entrenamiento[i].item()
    pred_clase = clases_entrenamiento[i].item()
    real_clase = y[i].item()

    marca = "OK" if pred_clase == real_clase else "REVISAR"
    sentimiento_real = "Positivo" if real_clase == 1 else "Negativo"
    sentimiento_pred = "Positivo" if pred_clase == 1 else "Negativo"

    frase_corta = frases[i]
    if len(frase_corta) > 55:
        frase_corta = frase_corta[:52] + "..."

    print(f"
{marca} Frase: '{frase_corta}'")
    print(f"  Real: {sentimiento_real} | Predicción: {sentimiento_pred} (prob={prob:.3f})")

print("
" + "=" * 70)
print(f"Accuracy de entrenamiento: {aciertos_entrenamiento}/{total_entrenamiento} = {accuracy_entrenamiento:.2%}")

frases_prueba_etiquetadas = [
    "La atención fue excelente y el lugar muy recomendable.",
    "Qué estafa, tardaron una banda y llegó todo frío.",
    "El servicio estuvo bien, pero no me dieron ganas de volver.",
    "Muy buena experiencia, volvería sin pensarlo."
]

etiquetas_prueba = [1, 0, 0, 1]
filas_prueba = []
for frase in frases_prueba_etiquetadas:
    fila_prueba = vectorizar(frase, vocabulario)
    filas_prueba.append(fila_prueba)

X_prueba_etiquetada_np = np.array(filas_prueba, dtype=np.float32)
X_prueba_etiquetada = torch.tensor(X_prueba_etiquetada_np)
y_prueba_etiquetada = torch.tensor(etiquetas_prueba, dtype=torch.float32).reshape(-1, 1)

with torch.no_grad():
    predicciones_prueba = modelo(X_prueba_etiquetada)
    clases_prueba = (predicciones_prueba >= 0.5).float()

aciertos_prueba = (clases_prueba == y_prueba_etiquetada).sum().item()
accuracy_prueba = aciertos_prueba / len(etiquetas_prueba)

print("
Prueba externa pequeña:")
print("=" * 70)
for i in range(len(frases_prueba_etiquetadas)):
    prob = predicciones_prueba[i].item()
    pred_clase = clases_prueba[i].item()
    real_clase = y_prueba_etiquetada[i].item()

    marca = "OK" if pred_clase == real_clase else "REVISAR"
    sentimiento_real = "Positivo" if real_clase == 1 else "Negativo"
    sentimiento_pred = "Positivo" if pred_clase == 1 else "Negativo"

    print(f"
{marca} Frase nueva: '{frases_prueba_etiquetadas[i]}'")
    print(f"  Real esperado: {sentimiento_real} | Predicción: {sentimiento_pred} (prob={prob:.3f})")

print("
" + "=" * 70)
print(f"Accuracy en prueba externa: {aciertos_prueba}/{len(etiquetas_prueba)} = {accuracy_prueba:.2%}")
print("
Lectura pedagógica: si la prueba externa cae, la red aprendió el corpus mejor de lo que generaliza fuera de él.")


---

## 12. Predicción sobre frases nuevas sin etiqueta

Ahora sí usamos el modelo como herramienta de exploración. Estas frases no tienen etiqueta cargada: el objetivo es mirar la probabilidad estimada y discutir si el resultado parece razonable.


In [ ]:
frases_prueba = [
    "No me gustó la atención, bastante mala",
    "Muy buena experiencia, todo excelente",
    "Una estafa total, no lo recomiendo",
    "Súper conforme con el servicio",
    "Nada que ver con lo prometido, una decepción",
    "La mejor atención que tuve en mucho tiempo"
]

filas_prueba = []
for frase in frases_prueba:
    fila_prueba = vectorizar(frase, vocabulario)
    filas_prueba.append(fila_prueba)

X_prueba_np = np.array(filas_prueba, dtype=np.float32)
X_prueba = torch.tensor(X_prueba_np)

modelo.eval()
with torch.no_grad():
    predicciones_prueba = modelo(X_prueba)

print("Predicciones sobre frases nuevas:")
print("=" * 70)

for i in range(len(frases_prueba)):
    frase = frases_prueba[i]
    prob = predicciones_prueba[i].item()
    clase = "POSITIVO" if prob >= 0.5 else "NEGATIVO"
    confianza = prob if prob >= 0.5 else (1 - prob)

    print(f"
Frase: '{frase}'")
    print(f"Predicción: {clase}")
    print(f"Probabilidad positivo: {prob:.3f}")
    print(f"Confianza: {confianza:.1%}")


---

## 13. Comparación: Perceptrón Simple vs. MLP

Comparemos conceptualmente lo que ganamos con capas ocultas.

In [ ]:
print("COMPARACIÓN: PERCEPTRÓN SIMPLE VS. MLP")
print("=" * 70)

comparacion = [
    ("Arquitectura", "Una capa (entrada → salida)", "Múltiples capas (entrada → ocultas → salida)"),
    ("Parámetros", "~30 (30 pesos + 1 bias)", "~280 (30×8 + 8 + 8×1 + 1)"),
    ("Capacidad", "Solo problemas lineales", "Problemas no lineales complejos"),
    ("Función activación", "Escalón (no diferenciable)", "ReLU + Sigmoid (diferenciables)"),
    ("Aprendizaje", "Regla del perceptrón", "Backpropagation con Adam"),
    ("XOR", " No puede resolverlo", "OK Sí puede resolverlo"),
    ("Interacciones", " No captura", "OK Aprende features abstractas"),
    ("Convergencia", "Rápida si linealmente separable", "Más lenta pero más potente"),
    ("Overfitting", "Poco riesgo (modelo simple)", "Mayor riesgo (más parámetros)"),
    ("Interpretabilidad", "Alta (pesos directos)", "Media (representaciones ocultas)"),
]

print(f"\n{'Característica':<20} | {'Perceptrón Simple':<35} | {'MLP':<35}")
print("-" * 95)
for caracteristica, perceptron, mlp in comparacion:
    print(f"{caracteristica:<20} | {perceptron:<35} | {mlp:<35}")

print("\n" + "=" * 70)
print("Conclusión:")
print("-" * 70)
print("El MLP es más potente pero requiere:")
print("  1. Más datos para entrenar (evitar overfitting)")
print("  2. Más tiempo de cómputo")
print("  3. Ajuste de hiperparámetros (learning rate, hidden size, etc.)")
print("  4. Frameworks modernos (PyTorch, TensorFlow) para eficiencia")
print("\nEl perceptrón simple sigue siendo útil como baseline rápido.")

---

## 14. Guardar y cargar el modelo (bloque opcional)

Este bloque no agrega teoría nueva sobre redes neuronales, pero muestra una práctica muy común de trabajo: persistir parámetros ya entrenados para reutilizarlos más tarde.


In [ ]:
torch.save(modelo.state_dict(), 'mlp_sentiment.pth')
print("Modelo guardado en: mlp_sentiment.pth")

modelo_cargado = MLP(input_size, hidden_size)
modelo_cargado.load_state_dict(torch.load('mlp_sentiment.pth'))
modelo_cargado.eval()

print("Modelo cargado correctamente.")

with torch.no_grad():
    pred_original = modelo(X_prueba)
    pred_cargado = modelo_cargado(X_prueba)
    igual = torch.allclose(pred_original, pred_cargado)

print(f"
¿Predicciones idénticas? {igual}")
print("En un proyecto real, este archivo permitiría reutilizar el modelo sin volver a entrenarlo desde cero.")


---

## Guía Teórico-Conceptual

### 1. Teorema de Aproximación Universal

**Enunciado (Cybenko, 1989; Hornik, 1991):**
> Una red neuronal feedforward con una sola capa oculta puede aproximar cualquier función continua en un dominio compacto, con precisión arbitraria, siempre que tenga suficientes neuronas ocultas.

**Implicaciones:**
- Los MLP son aproximadores universales de funciones
- En teoría, una capa oculta es suficiente
- En práctica, redes profundas (muchas capas) aprenden más eficientemente

**Limitaciones:**
- No dice cuántas neuronas se necesitan (podría ser exponencial)
- No garantiza que el entrenamiento encuentre la aproximación óptima
- Redes profundas suelen ser más eficientes que redes anchas

### 2. Backpropagation: La Magia de Autograd

Backpropagation es la aplicación de la regla de la cadena del cálculo:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial h} \cdot \frac{\partial h}{\partial w_1}$$

**En PyTorch:**
```python
loss.backward()  # Calcula TODOS los gradientes automáticamente
```

**Internamente:**
1. PyTorch registra todas las operaciones en un **grafo computacional**
2. Cuando llamamos `.backward()`, recorre el grafo en reversa
3. Aplica regla de la cadena en cada nodo
4. Acumula gradientes en `.grad` de cada tensor con `requires_grad=True`

**Ventaja sobre implementación manual:**
- No hay que derivar funciones a mano
- Funciona con arquitecturas arbitrariamente complejas
- Optimizado en C++/CUDA, muy rápido

### 3. ReLU vs. Sigmoid vs. Escalón

| Característica | Escalón | Sigmoid | ReLU |
|----------------|---------|---------|------|
| Fórmula | step(x) | 1/(1+e⁻ˣ) | max(0,x) |
| Rango | {0, 1} | (0, 1) | [0, ∞) |
| Diferenciable |  No | OK Sí | OK Casi (excepto x=0) |
| Vanishing gradient | N/A | OK Problema |  No sufre |
| Costo computacional | Bajo | Alto (exp) | Muy bajo |
| Uso moderno | Obsoleto | Solo salida | Capas ocultas |
| Dead neurons | No | No | Sí (si x<0 siempre) |

**¿Por qué ReLU domina en capas ocultas?**
- Gradiente constante para x>0 (no vanishing)
- Computacionalmente trivial (comparación)
- Introducida por Hinton et al. en 2010, revolucionó deep learning

**Variantes de ReLU:**
- **Leaky ReLU**: f(x) = max(0.01x, x) - evita dead neurons
- **ELU**: Suave en x<0
- **GELU**: Usada en transformers (BERT, GPT)

### 4. Vanishing Gradient Problem

**Problema:**
En redes profundas con sigmoides, los gradientes se multiplican capa por capa:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial h_n} \cdot \sigma'(h_{n-1}) \cdot ... \cdot \sigma'(h_2) \cdot \sigma'(h_1)$$

Como $\sigma'(x) \in (0, 0.25)$, el producto de muchos términos < 0.25 tiende a cero.

**Consecuencias:**
- Capas iniciales no aprenden (gradientes ~0)
- Solo capas finales se entrenan
- Redes profundas imposibles antes de 2010

**Soluciones:**
1. **ReLU**: Gradiente = 1 para x>0
2. **Batch Normalization**: Normaliza activaciones
3. **Residual connections**: Skip connections (ResNet)
4. **Inicialización cuidadosa**: Xavier, He initialization

### 5. Overfitting en Redes Neuronales

**Síntomas:**
- Training loss baja pero validation loss sube
- Accuracy muy alta en train, baja en test
- El modelo memoriza en vez de generalizar

**Causas:**
1. Demasiados parámetros vs. datos
2. Entrenamiento muy largo
3. Falta de regularización

**Soluciones:**

**Regularización L2 (weight decay):**
```python
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=0.01)
```

**Dropout:**
```python
nn.Dropout(p=0.5)  # Desactiva 50% de neuronas aleatoriamente
```

**Early stopping:**
```python
if val_loss > best_val_loss:
    epochs_without_improvement += 1
    if epochs_without_improvement > patience:
        break  # Detener entrenamiento
```

**Data augmentation:**
- Sinónimos, paráfrasis, traducciones

### 6. Hiperparámetros: Tuning Strategies

**Principales hiperparámetros:**
1. **Learning rate**: Más importante, probar [1e-5, 1e-1]
2. **Hidden size**: Capacidad del modelo
3. **Número de capas**: Profundidad
4. **Batch size**: Velocidad vs. convergencia
5. **Optimizer**: Adam suele ser la mejor opción

**Estrategias de búsqueda:**

**Grid search:**
```python
for lr in [0.001, 0.01, 0.1]:
    for hidden in [8, 16, 32]:
        train_and_evaluate(lr, hidden)
```

**Random search:** Más eficiente que grid

**Bayesian optimization:** Aún más eficiente

**Learning rate scheduling:**
```python
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.1)
```

### 7. Batch Training vs. Online Learning

En este notebook entrenamos con **full batch** (todo el dataset a la vez).

**Alternativas:**

**Stochastic Gradient Descent (SGD):**
- Una muestra a la vez
- Muy ruidoso pero escapa mínimos locales

**Mini-batch:**
- Típicamente 32-256 muestras
- Balance entre velocidad y estabilidad
- Estándar en deep learning

**Implementación:**
```python
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for X_batch, y_batch in dataloader:
    # Entrenar con el batch
```

---

## Preguntas y Respuestas para Estudio

### Preguntas Conceptuales

**1. ¿Por qué una red multicapa puede resolver XOR y el perceptrón simple no?**

*Respuesta:* La capa oculta proyecta los datos a un espacio de mayor dimensión donde SÍ son linealmente separables. Por ejemplo, XOR en 2D no es separable, pero si la capa oculta aprende las features h₁=(x₁ AND x₂) y h₂=(x₁ OR x₂), entonces en el espacio (h₁, h₂) SÍ es separable. La capa oculta aprende representaciones útiles automáticamente.

**2. ¿Qué es autograd y cómo funciona en PyTorch?**

*Respuesta:* Autograd (automatic differentiation) es el sistema de PyTorch que calcula gradientes automáticamente. Registra todas las operaciones en un grafo computacional y, cuando llamamos `loss.backward()`, recorre el grafo en reversa aplicando la regla de la cadena. Esto nos libera de derivar funciones manualmente.

**3. ¿Por qué usamos ReLU en capas ocultas y Sigmoid en la salida?**

*Respuesta:*
- **ReLU en ocultas**: No sufre vanishing gradient, es computacionalmente eficiente y permite entrenar redes profundas
- **Sigmoid en salida**: Convierte la salida a rango [0,1] interpretable como probabilidad, compatible con Binary Cross Entropy Loss

**4. ¿Qué significa "requires_grad=True" en un tensor?**

*Respuesta:* Indica que PyTorch debe calcular y almacenar gradientes para ese tensor. Los parámetros del modelo (pesos, biases) tienen `requires_grad=True` por defecto. Los datos de entrada no, porque no los vamos a optimizar.

**5. ¿Por qué llamamos `optimizer.zero_grad()` antes de cada backward pass?**

*Respuesta:* Porque PyTorch **acumula** gradientes por defecto (suma los nuevos a los existentes). Si no limpiamos, estaríamos acumulando gradientes de múltiples batches, lo cual es incorrecto. En algunos casos avanzados (gradient accumulation) esta acumulación es intencional, pero normalmente queremos empezar de cero.

### Preguntas Técnicas

**6. ¿Cuál es la diferencia entre `model.train()` y `model.eval()`?**

*Respuesta:*
- `model.train()`: Activa modo entrenamiento. Dropout hace drop, BatchNorm actualiza estadísticas
- `model.eval()`: Activa modo evaluación. Dropout no hace drop, BatchNorm usa estadísticas fijas

Aunque nuestro MLP no usa estas capas, es buena práctica llamarlos para compatibilidad.

**7. ¿Qué hace `torch.no_grad()` y cuándo usarlo?**

*Respuesta:* Desactiva el tracking de gradientes. Útil durante evaluación/inferencia para:
1. Ahorrar memoria (no construir grafo computacional)
2. Acelerar ejecución
3. Prevenir errores (actualizar parámetros accidentalmente)

**8. En el código, ¿por qué reshape(-1, 1) en las etiquetas?**

*Respuesta:* `BCELoss` espera que predicciones y etiquetas tengan la misma forma. Nuestro modelo devuelve shape `(n, 1)`, así que las etiquetas también deben ser `(n, 1)` en vez de `(n,)`. El `-1` en reshape significa "inferir esta dimensión automáticamente".

**9. ¿Qué es un "forward pass" y un "backward pass"?**

*Respuesta:*
- **Forward pass**: Los datos fluyen de entrada a salida, calculando predicciones: `y_pred = model(X)`
- **Backward pass**: Los gradientes fluyen de salida a entrada, calculando ∂Loss/∂params: `loss.backward()`

**10. ¿Por qué `.item()` en `loss.item()`?**

*Respuesta:* `.item()` extrae el valor numérico de un tensor de un solo elemento y lo convierte a tipo Python (float). Sin `.item()`, tendríamos un tensor, que consume memoria del grafo computacional. Es buena práctica para logging.

### Preguntas de Aplicación

**11. Si la curva de loss oscila mucho sin bajar, ¿qué harías?**

*Respuesta:*
1. **Reducir learning rate**: Probar 0.001 en vez de 0.01
2. **Cambiar optimizador**: Adam → SGD with momentum
3. **Normalizar datos**: Estandarizar features
4. **Revisar implementación**: ¿Está llamando zero_grad()?
5. **Aumentar batch size**: Más estabilidad (si usás mini-batches)

**12. ¿Cómo adaptarías este código para clasificación multiclase (3+ clases)?**

*Respuesta:*
```python
# 1. Cambiar salida del modelo
nn.Linear(hidden_size, num_classes),  # En vez de 1
nn.Softmax(dim=1)  # En vez de Sigmoid

# 2. Cambiar loss
criterio = nn.CrossEntropyLoss()  # En vez de BCELoss

# 3. Etiquetas como enteros
y = torch.tensor([0, 1, 2, 0, 1, ...])  # Sin one-hot

# 4. Predicción
pred_class = torch.argmax(output, dim=1)  # Clase con mayor prob
```

**13. Si tuvieras 10,000 frases en vez de 20, ¿qué cambios harías al código?**

*Respuesta:*
1. **Usar DataLoader** para mini-batches (batch_size=32 o 64)
2. **Split train/test**: 80/20 para evaluación honesta
3. **Más épocas**: Probablemente 50-200 en vez de 200 sobre dataset pequeño
4. **Red más grande**: hidden_size=64 o 128
5. **Agregar Dropout**: Para prevenir overfitting
6. **Usar GPU**: `model.to('cuda')` para acelerar

**14. ¿Cómo implementarías early stopping?**

*Respuesta:*
```python
best_val_loss = float('inf')
patience = 10
epochs_without_improvement = 0

for epoca in range(max_epocas):
    # Entrenar...
    val_loss = evaluar_en_validacion()
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    if epochs_without_improvement >= patience:
        print("Early stopping")
        break
```

**15. Explicá cómo desplegarías este modelo en una API REST.**

*Respuesta:*
```python
from fastapi import FastAPI
import torch

app = FastAPI()

# Cargar modelo al inicio
modelo = MLP(input_size=30, hidden_size=8)
modelo.load_state_dict(torch.load('mlp_sentiment.pth'))
modelo.eval()

@app.post("/predict")
def predict(texto: str):
    # Vectorizar
    x = torch.tensor(vectorizar(texto, vocabulario))
    
    # Predecir
    with torch.no_grad():
        prob = modelo(x.unsqueeze(0)).item()
    
    return {
        "texto": texto,
        "sentimiento": "positivo" if prob > 0.5 else "negativo",
        "probabilidad": prob
    }
```

---

## Ejercicios propuestos

### Ejercicio 1: tamaño de la capa oculta
Probá `hidden_size = 4`, `8`, `16` y `32`. Compará la curva de pérdida y la pequeña prueba externa.

### Ejercicio 2: funciones de activación
Reemplazá `ReLU` por `Tanh` o `LeakyReLU` y observá si cambia la estabilidad del entrenamiento.

### Ejercicio 3: regularización
Agregá una capa `Dropout` después de la activación oculta y discutí en qué sentido podría ayudar cuando el corpus crece.

### Ejercicio 4: problema XOR
Usá este mismo modelo para resolver XOR y compará el resultado con el perceptrón simple.

## Cierre

El MLP conserva la idea central del perceptrón, pero suma un elemento decisivo: una capa oculta con no linealidad. Ese cambio es el que permite salir del mundo estrictamente lineal.

En el próximo notebook vamos a trabajar con secuencias y memoria temporal mediante LSTM.
